# JeevaSwara / KanthaRakshak: 02. Signal Processing & Event Detection

In this notebook, we evaluate:
1. Signal conditioning (DC removal, bandpass filtering, envelope extraction)
2. Accelerometer dynamic gravity removal and jerk computation
3. Dual-sensor coincidence swallow event detection


In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath(".."))
from app.pipeline.data_format import TelemetrySession, SessionMetadata, SwallowEvent
from app.pipeline.preprocessing import (
    remove_dc_offset, bandpass_filter, smooth_envelope, estimate_noise_floor
)
from app.pipeline.accelerometer import (
    calculate_accel_magnitude, remove_gravity_baseline, compute_motion_envelope, compute_jerk
)
from app.pipeline.event_detection import detect_swallow_event

# Synthesize a demonstration normal swallow session
fs = 50.0
t = np.arange(200) * (1000.0 / fs) # 4 seconds at 50Hz
piezo = np.random.normal(0, 0.015, 200)
# Swallow burst from 1400ms to 2100ms
piezo[70:105] += 0.65 * np.sin(np.pi * np.linspace(0, 1, 35))**2

ax = np.random.normal(0, 0.01, 200)
ax[72:107] += 0.28 * np.sin(np.pi * np.linspace(0, 1, 35))
ay = np.zeros(200)
az = np.random.normal(0.98, 0.01, 200)

session = TelemetrySession(timestamp_ms=t, piezo=piezo, ax=ax, ay=ay, az=az)
event = detect_swallow_event(session)
print("Detected Event:", event)


### Preprocessing Visualization
Let us plot the raw piezo signal, the bandpassed acoustic energy, the accelerometer magnitude, and the isolated event boundary.


In [ ]:
# Preprocessing steps
piezo_clean = remove_dc_offset(session.piezo)
piezo_filtered = bandpass_filter(piezo_clean, fs=50.0, lowcut=5.0, highcut=20.0)
piezo_env = smooth_envelope(piezo_clean, window_len=7)

accel_mag = calculate_accel_magnitude(session.ax, session.ay, session.az)
dynamic_motion = remove_gravity_baseline(accel_mag)
motion_env = compute_motion_envelope(dynamic_motion, window_size=7)

fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)

# 1. Acoustic Piezo
axes[0].plot(t, session.piezo, label='Raw Piezo', color='#94a3b8', alpha=0.7)
axes[0].plot(t, piezo_env, label='Smoothed Acoustic Envelope', color='#0d9488', lw=2)
if isinstance(event, SwallowEvent):
    axes[0].axvspan(event.event_start, event.event_end, color='#fef08a', alpha=0.35, label='Detected Swallow Window')
axes[0].set_ylabel('Amplitude (V / a.u.)')
axes[0].set_title('27mm Throat Piezoelectric Contact Microphone')
axes[0].legend(loc='upper right')

# 2. Kinematic Accelerometer
axes[1].plot(t, accel_mag, label='Total Magnitude (g)', color='#64748b', alpha=0.6)
axes[1].plot(t, motion_env, label='Motion Excursion Envelope', color='#0284c7', lw=2)
if isinstance(event, SwallowEvent):
    axes[1].axvspan(event.event_start, event.event_end, color='#fef08a', alpha=0.35)
axes[1].set_ylabel('Acceleration (g)')
axes[1].set_title('MPU6050 Cervical Motion & Hyolaryngeal Excursion')
axes[1].legend(loc='upper right')

# 3. Jerk (Rate of change of acceleration)
jerk = compute_jerk(accel_mag, dt=0.02)
axes[2].plot(t, jerk, label='Kinematic Jerk (g/s)', color='#e11d48', lw=1.5)
axes[2].set_ylabel('Jerk (g/s)')
axes[2].set_xlabel('Timestamp (ms)')
axes[2].set_title('Derivative Kinematic Jerk')
axes[2].legend(loc='upper right')

plt.tight_layout()
plt.show()


### Signal Processing Observations
- Dual-sensor coincidence detection effectively isolates the swallow window while discarding respiratory baseline tremor.
- The temporal lag between the acoustic peak and the kinematic motion peak is ~40 ms, characteristic of coordinated submental elevation.
